# CineMatch: audit, global-time split, and cross-language test

In [1]:
# Set these True only after inspecting the dry-run summaries.
WRITE_GLOBAL_SPLITS = True
WRITE_LEGACY_USERWISE_SPLIT = False

SEED = 42
POSITIVE_THRESHOLD = 4.0
MIN_TRAIN_POSITIVES = 5
MIN_KNOWN_HISTORY_POSITIVES = 10
MIN_DOMINANT_LANGUAGE_SHARE = 0.60
PRIMARY_USER_SAMPLE = 2500  # uniform random sample; set None to evaluate every eligible user
MAX_TARGETS_PER_USER = 10   # earliest positive events in the future window, fixed before model evaluation
USERS_PER_ACTIVITY_BAND = 200  # secondary balanced activity analysis only
DEVELOPMENT_QUANTILE = 0.80
VALIDATION_QUANTILE = 0.90
HASH_RAW_INPUTS = True
CHECK_DUPLICATE_USER_ITEMS = True

from pathlib import Path
from collections import Counter
import hashlib
import json
import numpy as np
import pandas as pd
from IPython.display import display
import scipy.sparse as sp

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CANDIDATES = [
    Path('/content/drive/MyDrive/cinematch'),
    Path('/blue/egn6933/nagabhairava.r'),
    Path('.').resolve(),
]

def resolve_base():
    for base in CANDIDATES:
        if (base / 'Data/ml-32m/ratings.csv').exists() and (base / 'Data/ml-32m/links.csv').exists():
            return base
    raise FileNotFoundError('Stage Data/ml-32m/ratings.csv and links.csv under a candidate project root.')

BASE = resolve_base()
OUT = BASE / 'outputs/publication_audit'
OUT.mkdir(parents=True, exist_ok=True)
RATINGS = BASE / 'Data/ml-32m/ratings.csv'
LINKS = BASE / 'Data/ml-32m/links.csv'
OLD_HOLDOUT = BASE / 'outputs/xsimgcl/test_holdout.csv'
catalog_paths = [
    BASE / 'Data/tmdb_semantic_catalog_alllangs_with_new_movies.csv',
    BASE / 'Data/outputs/tmdb_semantic_catalog_alllangs_with_new_movies.csv',
]
CATALOG = next((path for path in catalog_paths if path.exists()), None)
if CATALOG is None:
    raise FileNotFoundError('Stage the TMDB catalog with id and original_language columns before running this audit.')
print('Base:', BASE)
print('Output:', OUT)

Mounted at /content/drive
Base: /content/drive/MyDrive/cinematch
Output: /content/drive/MyDrive/cinematch/outputs/publication_audit


In [2]:
# Load the language map once. Missing mappings remain missing; they are never silently treated as English.
dtype = {'userId': 'int32', 'movieId': 'int32', 'rating': 'float32', 'timestamp': 'int32'}
links = pd.read_csv(LINKS, usecols=['movieId', 'tmdbId'])
links.tmdbId = pd.to_numeric(links.tmdbId, errors='coerce')
ml_to_tmdb = {int(movie): int(tmdb) for movie, tmdb in zip(links.movieId, links.tmdbId) if pd.notna(tmdb)}
# Reverse lookup is needed when semantic FAISS results return TMDB-linked titles.
tmdb_to_ml = {tmdb_id: movie_id for movie_id, tmdb_id in ml_to_tmdb.items()}
catalog = pd.read_csv(CATALOG, usecols=['id', 'original_language'], low_memory=False)
catalog.id = pd.to_numeric(catalog.id, errors='coerce')
catalog = catalog.dropna(subset=['id']).drop_duplicates('id').set_index('id')
tmdb_to_lang = catalog.original_language.astype('string').str.strip().str.lower().to_dict()

def language_for_movie(movie_id):
    tmdb_id = ml_to_tmdb.get(int(movie_id))
    language = tmdb_to_lang.get(tmdb_id) if tmdb_id is not None else None
    if language is None or pd.isna(language):
        return None
    language = str(language).strip().lower()
    return language if language and language not in {'nan', '<na>'} else None

old_holdout = None
if OLD_HOLDOUT.exists():
    old_holdout = pd.read_csv(OLD_HOLDOUT, dtype=dtype)
    assert (old_holdout.rating >= POSITIVE_THRESHOLD).all()
    assert old_holdout.groupby('userId').size().eq(10).all()
    audit = old_holdout.copy()
    audit['original_language'] = audit.movieId.map(language_for_movie).fillna('unmapped')
    language_counts = audit.original_language.value_counts().rename_axis('language').reset_index(name='target_events')
    language_counts['share_of_all_targets'] = language_counts.target_events / len(audit)
    language_counts.to_csv(OUT / 'legacy_positive_event_target_language_audit.csv', index=False)
    known = audit.original_language.ne('unmapped')
    legacy_language_summary = {
        'users': int(audit.userId.nunique()),
        'positive_target_events': int(len(audit)),
        'known_language_coverage': float(known.mean()),
        'english_share_of_all_targets': float(audit.original_language.eq('en').mean()),
        'english_share_of_known_targets': float(audit.loc[known, 'original_language'].eq('en').mean()) if known.any() else None,
    }
    with (OUT / 'legacy_positive_event_language_summary.json').open('w') as f:
        json.dump(legacy_language_summary, f, indent=2)
    display(language_counts.head(20))
    print(json.dumps(legacy_language_summary, indent=2))
    print('These are title-language shares only, never user-culture labels.')
else:
    print('Legacy holdout not found; skipping legacy audit and continuing with the global-time protocol.')

,language,target_events,share_of_all_targets
0,en,8271,0.844842
1,unmapped,859,0.087743
2,ja,159,0.016241
3,fr,144,0.014709
4,es,75,0.007661
5,it,60,0.006129
6,de,49,0.005005
7,cn,23,0.002349
8,zh,22,0.002247
9,ko,22,0.002247


{
  "users": 979,
  "positive_target_events": 9790,
  "known_language_coverage": 0.9122574055158325,
  "english_share_of_all_targets": 0.8448416751787539,
  "english_share_of_known_targets": 0.9261001007725899
}
These are title-language shares only, never user-culture labels.


In [3]:
# Read the interaction log once. The legacy audit below documents why its positive-event holdout is not temporal.
ratings = pd.read_csv(RATINGS, dtype=dtype)
print(f'Loaded {len(ratings):,} ratings from {RATINGS}')
if CHECK_DUPLICATE_USER_ITEMS:
    duplicate_pairs = int(ratings.duplicated(['userId', 'movieId'], keep=False).sum())
    assert duplicate_pairs == 0, f'Found {duplicate_pairs:,} duplicate user-item rows; define a deterministic deduplication rule before splitting.'

if old_holdout is not None:
    cutoff_by_user = old_holdout.groupby('userId').timestamp.min().astype('int64').to_dict()
    selected = ratings.userId.isin(cutoff_by_user)
    future_or_tied = selected & ratings.timestamp.ge(ratings.userId.map(cutoff_by_user))
    future = ratings.loc[future_or_tied, ['userId', 'movieId', 'rating', 'timestamp']]
    old_keys = set(zip(old_holdout.userId.astype(int), old_holdout.movieId.astype(int), old_holdout.timestamp.astype(int)))
    future_keys = set(zip(future.userId.astype(int), future.movieId.astype(int), future.timestamp.astype(int)))
    leakage = {
        'selected_users': int(len(cutoff_by_user)),
        'legacy_positive_targets': int(len(old_holdout)),
        'all_events_at_or_after_per_user_cutoff': int(len(future)),
        'non_target_future_events_left_in_legacy_training': int(len(future_keys - old_keys)),
    }
    with (OUT / 'legacy_positive_event_leakage_audit.json').open('w') as f:
        json.dump(leakage, f, indent=2)
    print(json.dumps(leakage, indent=2))

    if WRITE_LEGACY_USERWISE_SPLIT:
        strict_train = ratings.loc[~future_or_tied]
        strict_targets = future.loc[future.rating >= POSITIVE_THRESHOLD]
        assert not (strict_train.userId.isin(cutoff_by_user) & strict_train.timestamp.ge(strict_train.userId.map(cutoff_by_user))).any()
        strict_train.to_csv(OUT / 'legacy_userwise_strict_train_ratings.csv', index=False)
        strict_targets.to_csv(OUT / 'legacy_userwise_strict_positive_targets.csv', index=False)
        print('Wrote optional legacy user-wise strict split.')
    else:
        print('Legacy strict split not written (set WRITE_LEGACY_USERWISE_SPLIT=True only if you need a legacy-cohort diagnostic).')

Loaded 32,000,204 ratings from /content/drive/MyDrive/cinematch/Data/ml-32m/ratings.csv
{
  "selected_users": 979,
  "legacy_positive_targets": 9790,
  "all_events_at_or_after_per_user_cutoff": 28680,
  "non_target_future_events_left_in_legacy_training": 18890
}
Legacy strict split not written (set WRITE_LEGACY_USERWISE_SPLIT=True only if you need a legacy-cohort diagnostic).


In [4]:
# Primary publication split: global chronological development / validation / test windows.
# A global cutoff prevents any method from learning interaction data from a later calendar period.
def quantile_timestamp(values, q):
    try:
        return int(np.quantile(values, q, method='lower'))
    except TypeError:  # NumPy < 1.22
        return int(np.quantile(values, q, interpolation='lower'))

dev_cutoff = quantile_timestamp(ratings.timestamp.to_numpy(), DEVELOPMENT_QUANTILE)
validation_cutoff = quantile_timestamp(ratings.timestamp.to_numpy(), VALIDATION_QUANTILE)
assert dev_cutoff < validation_cutoff
dev_mask = ratings.timestamp.le(dev_cutoff)
validation_mask = ratings.timestamp.gt(dev_cutoff) & ratings.timestamp.le(validation_cutoff)
final_train_mask = ratings.timestamp.le(validation_cutoff)
test_mask = ratings.timestamp.gt(validation_cutoff)
assert not (dev_mask & validation_mask).any()
assert not (final_train_mask & test_mask).any()

BINS = [MIN_TRAIN_POSITIVES - 1, 50, 100, 200, 500, np.inf]
LABELS = ['5-50', '50-100', '100-200', '200-500', '500+']

def build_fixed_cohort(history_mask, target_mask, candidate_items, seed):
    # Primary users are sampled uniformly from all eligible users. Activity-balanced users are saved separately.
    # Candidate-cold positives are retained for a prespecified zero-shot secondary analysis.
    history_positive = ratings.loc[history_mask & ratings.rating.ge(POSITIVE_THRESHOLD), ['userId', 'movieId']]
    history_counts = history_positive.groupby('userId').size()
    future_positive_all = ratings.loc[
        target_mask & ratings.rating.ge(POSITIVE_THRESHOLD),
        ['userId', 'movieId', 'rating', 'timestamp']
    ].copy()
    known_positive = future_positive_all.loc[future_positive_all.movieId.isin(candidate_items)].copy()
    cold_positive = future_positive_all.loc[~future_positive_all.movieId.isin(candidate_items)].copy()
    eligible = history_counts[history_counts.ge(MIN_TRAIN_POSITIVES)].index.intersection(
        known_positive.userId.unique()
    )
    rng = np.random.default_rng(seed)
    sample_n = len(eligible) if PRIMARY_USER_SAMPLE is None else min(PRIMARY_USER_SAMPLE, len(eligible))
    selected = sorted(rng.choice(eligible.to_numpy(), size=sample_n, replace=False).astype('int64').tolist())

    activity = history_counts.loc[eligible]
    bands = pd.cut(activity, bins=BINS, labels=LABELS)
    balanced = []
    for label in LABELS:
        members = bands[(bands == label) & bands.index.isin(selected)].index.to_numpy()
        take = min(USERS_PER_ACTIVITY_BAND, len(members))
        if take:
            balanced.extend(rng.choice(members, size=take, replace=False).astype('int64').tolist())
    balanced = sorted(set(map(int, balanced)))

    cohort_targets = (known_positive.loc[known_positive.userId.isin(selected)]
                      .sort_values(['userId', 'timestamp', 'movieId'])
                      .groupby('userId', group_keys=False).head(MAX_TARGETS_PER_USER).copy())
    cold_eligible = history_counts[history_counts.ge(MIN_TRAIN_POSITIVES)].index.intersection(
        cold_positive.userId.unique()
    )
    cold_n = len(cold_eligible) if PRIMARY_USER_SAMPLE is None else min(PRIMARY_USER_SAMPLE, len(cold_eligible))
    cold_users = sorted(rng.choice(cold_eligible.to_numpy(), size=cold_n, replace=False).astype('int64').tolist()) if cold_n else []
    cold_targets = (cold_positive.loc[cold_positive.userId.isin(cold_users)]
                    .sort_values(['userId', 'timestamp', 'movieId'])
                    .groupby('userId', group_keys=False).head(MAX_TARGETS_PER_USER).copy())
    selected_activity = pd.cut(history_counts.loc[selected], bins=BINS, labels=LABELS).value_counts().reindex(LABELS, fill_value=0)
    return selected, cohort_targets, balanced, cold_users, cold_targets, {
        'eligible_users_before_sampling': int(len(eligible)),
        'sampled_users': int(len(selected)),
        'sampled_activity_distribution': {str(k): int(v) for k, v in selected_activity.items()},
        'balanced_secondary_users': int(len(balanced)),
        'positive_targets_in_primary_cohort': int(len(cohort_targets)),
        'candidate_cold_eligible_users': int(len(cold_eligible)),
        'candidate_cold_sampled_users': int(len(cold_users)),
        'candidate_cold_targets': int(len(cold_targets)),
    }

development_candidates = set(ratings.loc[dev_mask, 'movieId'].astype(int).unique())
final_candidates = set(ratings.loc[final_train_mask, 'movieId'].astype(int).unique())
validation_users, validation_targets, validation_balanced_users, validation_cold_users, validation_cold_targets, validation_summary = build_fixed_cohort(
    dev_mask, validation_mask, development_candidates, SEED
)
test_users, test_targets, test_balanced_users, test_cold_users, test_cold_targets, test_summary = build_fixed_cohort(
    final_train_mask, test_mask, final_candidates, SEED + 1
)

# Executable leakage/overlap invariants. A failed assertion invalidates the run.
assert test_targets.timestamp.gt(validation_cutoff).all()
assert validation_targets.timestamp.gt(dev_cutoff).all() and validation_targets.timestamp.le(validation_cutoff).all()
assert set(test_targets.movieId.astype(int)).issubset(final_candidates)
assert set(validation_targets.movieId.astype(int)).issubset(development_candidates)
assert set(test_cold_targets.movieId.astype(int)).isdisjoint(final_candidates)
assert set(test_targets.userId.astype(int)).issubset(set(test_users))
assert set(validation_targets.userId.astype(int)).issubset(set(validation_users))
train_pairs = pd.MultiIndex.from_frame(ratings.loc[final_train_mask, ['userId', 'movieId']])
test_pairs = pd.MultiIndex.from_frame(test_targets[['userId', 'movieId']])
assert len(train_pairs.intersection(test_pairs)) == 0

manifest = {
    'protocol': 'global chronological split with frozen candidate catalog',
    'seed': SEED,
    'positive_threshold': POSITIVE_THRESHOLD,
    'minimum_train_positives': MIN_TRAIN_POSITIVES,
    'primary_user_sample': PRIMARY_USER_SAMPLE,
    'maximum_targets_per_user': MAX_TARGETS_PER_USER,
    'users_per_activity_band_cap': USERS_PER_ACTIVITY_BAND,
    'development_cutoff_unix': dev_cutoff,
    'validation_cutoff_unix': validation_cutoff,
    'development_cutoff_utc': pd.to_datetime(dev_cutoff, unit='s', utc=True).isoformat(),
    'validation_cutoff_utc': pd.to_datetime(validation_cutoff, unit='s', utc=True).isoformat(),
    'development_train_rows': int(dev_mask.sum()),
    'validation_window_rows': int(validation_mask.sum()),
    'final_train_rows': int(final_train_mask.sum()),
    'test_window_rows': int(test_mask.sum()),
    'development_candidate_items': int(len(development_candidates)),
    'final_candidate_items': int(len(final_candidates)),
    'validation': validation_summary,
    'test': test_summary,
}
with (OUT / 'global_time_split_manifest.json').open('w') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

if HASH_RAW_INPUTS:
    manifest['input_sha256'] = {
        'ratings.csv': sha256_file(RATINGS),
        'links.csv': sha256_file(LINKS),
        CATALOG.name: sha256_file(CATALOG),
    }
    with (OUT / 'global_time_split_manifest.json').open('w') as f:
        json.dump(manifest, f, indent=2)
    print('Recorded raw-input SHA-256 fingerprints.')

paths_to_write = {
    'development_train': OUT / 'global_time_development_train_ratings.csv',
    'validation_targets': OUT / 'global_time_validation_positive_targets.csv',
    'validation_users': OUT / 'global_time_validation_users.csv',
    'development_candidate_items': OUT / 'global_time_development_candidate_movie_ids.csv',
    'final_train': OUT / 'global_time_final_train_ratings.csv',
    'test_targets': OUT / 'global_time_test_positive_targets.csv',
    'test_users': OUT / 'global_time_test_users.csv',
    'balanced_test_users': OUT / 'global_time_balanced_test_users.csv',
    'candidate_items': OUT / 'global_time_final_candidate_movie_ids.csv',
    'cold_test_targets': OUT / 'global_time_cold_item_positive_targets.csv',
    'cold_test_users': OUT / 'global_time_cold_item_users.csv',
}
if WRITE_GLOBAL_SPLITS:
    ratings.loc[dev_mask].to_csv(paths_to_write['development_train'], index=False)
    validation_targets.to_csv(paths_to_write['validation_targets'], index=False)
    pd.DataFrame({'userId': validation_users}).to_csv(paths_to_write['validation_users'], index=False)
    pd.DataFrame({'movieId': sorted(development_candidates)}).to_csv(paths_to_write['development_candidate_items'], index=False)
    ratings.loc[final_train_mask].to_csv(paths_to_write['final_train'], index=False)
    test_targets.to_csv(paths_to_write['test_targets'], index=False)
    pd.DataFrame({'userId': test_users}).to_csv(paths_to_write['test_users'], index=False)
    pd.DataFrame({'userId': test_balanced_users}).to_csv(paths_to_write['balanced_test_users'], index=False)
    pd.DataFrame({'movieId': sorted(final_candidates)}).to_csv(paths_to_write['candidate_items'], index=False)
    test_cold_targets.to_csv(paths_to_write['cold_test_targets'], index=False)
    pd.DataFrame({'userId': test_cold_users}).to_csv(paths_to_write['cold_test_users'], index=False)
    for name, path in paths_to_write.items():
        manifest.setdefault('sha256', {})[name] = sha256_file(path)
    with (OUT / 'global_time_split_manifest.json').open('w') as f:
        json.dump(manifest, f, indent=2)
    print('Wrote global chronological split files and SHA-256 manifest.')
else:
    print('Dry run complete. Inspect this manifest, then set WRITE_GLOBAL_SPLITS=True and rerun this cell to write the split files.')

{
  "protocol": "global chronological split with frozen candidate catalog",
  "seed": 42,
  "positive_threshold": 4.0,
  "minimum_train_positives": 5,
  "primary_user_sample": 2500,
  "maximum_targets_per_user": 10,
  "users_per_activity_band_cap": 200,
  "development_cutoff_unix": 1538551300,
  "validation_cutoff_unix": 1604605533,
  "development_cutoff_utc": "2018-10-03T07:21:40+00:00",
  "validation_cutoff_utc": "2020-11-05T19:45:33+00:00",
  "development_train_rows": 25600163,
  "validation_window_rows": 3200020,
  "final_train_rows": 28800183,
  "test_window_rows": 3200021,
  "development_candidate_items": 50977,
  "final_candidate_items": 65723,
  "validation": {
    "eligible_users_before_sampling": 6330,
    "sampled_users": 2500,
    "sampled_activity_distribution": {
      "5-50": 469,
      "50-100": 480,
      "100-200": 660,
      "200-500": 682,
      "500+": 209
    },
    "balanced_secondary_users": 1000,
    "positive_targets_in_primary_cohort": 19019,
    "candidate_c

In [5]:
# Secondary cross-language retrieval subset from the *primary global test cohort*.
# Eligibility is declared without looking at any model output. A title's original language is not a user's culture.
history = ratings.loc[
    final_train_mask & ratings.userId.isin(test_users) & ratings.rating.ge(POSITIVE_THRESHOLD),
    ['userId', 'movieId']
].copy()
history['language'] = history.movieId.map(language_for_movie)
history = history.dropna(subset=['language'])

dominant_language = {}
known_history_count = {}
dominant_language_share = {}
for user_id, group in history.groupby('userId'):
    counts = Counter(group.language)
    total = int(sum(counts.values()))
    top_count = max(counts.values())
    leaders = [language for language, count in counts.items() if count == top_count]
    known_history_count[int(user_id)] = total
    if len(leaders) == 1 and total >= MIN_KNOWN_HISTORY_POSITIVES and top_count / total >= MIN_DOMINANT_LANGUAGE_SHARE:
        dominant_language[int(user_id)] = leaders[0]
        dominant_language_share[int(user_id)] = float(top_count / total)

cross_targets = test_targets.copy()
cross_targets['target_language'] = cross_targets.movieId.map(language_for_movie)
cross_targets['history_dominant_language'] = cross_targets.userId.map(dominant_language)
cross_targets['known_history_positive_count'] = cross_targets.userId.map(known_history_count).fillna(0).astype(int)
cross_targets['history_dominant_language_share'] = cross_targets.userId.map(dominant_language_share)
cross_targets = cross_targets.loc[
    cross_targets.target_language.notna()
    & cross_targets.history_dominant_language.notna()
    & cross_targets.target_language.ne(cross_targets.history_dominant_language)
    & cross_targets.known_history_positive_count.ge(MIN_KNOWN_HISTORY_POSITIVES)
    & cross_targets.history_dominant_language_share.ge(MIN_DOMINANT_LANGUAGE_SHARE)
].copy()
cross_summary = {
    'definition': 'future positive title language differs from dominant mapped positive language in final training history',
    'eligible_users': int(cross_targets.userId.nunique()),
    'cross_language_positive_targets': int(len(cross_targets)),
    'minimum_known_history_positives': MIN_KNOWN_HISTORY_POSITIVES,
    'minimum_dominant_language_share': MIN_DOMINANT_LANGUAGE_SHARE,
    'language_mapping_coverage_of_primary_test_targets': float(test_targets.movieId.map(language_for_movie).notna().mean()),
    'direction_counts': {str(k): int(v) for k, v in cross_targets.groupby(['history_dominant_language', 'target_language']).size().items()},
}
cross_targets.to_csv(OUT / 'global_time_cross_language_positive_targets.csv', index=False)
with (OUT / 'global_time_cross_language_summary.json').open('w') as f:
    json.dump(cross_summary, f, indent=2)
print(json.dumps(cross_summary, indent=2))
display(cross_targets[['userId', 'movieId', 'history_dominant_language', 'target_language']].head(20))
print('Report this only as secondary cross-language retrieval evidence; it is not a culture, fairness, or satisfaction result.')

{
  "definition": "future positive title language differs from dominant mapped positive language in final training history",
  "eligible_users": 1013,
  "cross_language_positive_targets": 2073,
  "minimum_known_history_positives": 10,
  "minimum_dominant_language_share": 0.6,
  "language_mapping_coverage_of_primary_test_targets": 0.8712601060816726,
  "direction_counts": {
    "('en', 'af')": 1,
    "('en', 'ar')": 17,
    "('en', 'bg')": 1,
    "('en', 'bn')": 4,
    "('en', 'bs')": 1,
    "('en', 'cn')": 49,
    "('en', 'cs')": 7,
    "('en', 'da')": 32,
    "('en', 'de')": 115,
    "('en', 'dz')": 2,
    "('en', 'el')": 12,
    "('en', 'es')": 173,
    "('en', 'et')": 5,
    "('en', 'fa')": 7,
    "('en', 'fi')": 17,
    "('en', 'fr')": 355,
    "('en', 'he')": 4,
    "('en', 'hi')": 43,
    "('en', 'hu')": 10,
    "('en', 'id')": 5,
    "('en', 'is')": 1,
    "('en', 'it')": 142,
    "('en', 'iu')": 1,
    "('en', 'ja')": 581,
    "('en', 'ka')": 1,
    "('en', 'ko')": 152,
    "('

,userId,movieId,history_dominant_language,target_language
50488,326,26326,en,es
145655,947,202749,en,fr
163831,1050,224208,en,sr
170679,1089,193455,en,sv
170530,1089,114554,en,ja
176784,1140,8014,en,ko
212884,1399,5618,en,ja
255263,1666,31364,en,ko
284418,1819,114554,en,ja
284331,1819,26776,en,ja


Report this only as secondary cross-language retrieval evidence; it is not a culture, fairness, or satisfaction result.


## Phase II — self-contained model retraining and evaluation

The remaining cells are resume-safe and write every artifact under `outputs/publication_audit/publication_eval`. They train development BPR, LightGCN, and XSimGCL models for epoch selection, then refit each model on every pre-test positive interaction before evaluating the untouched test period.

The primary test is the globally later, known-item cohort. Cross-language retrieval and item cold-start are prespecified secondary tests. Current IMDb rating/vote metadata is evaluated only in a clearly labeled snapshot ablation because historical IMDb vote snapshots are unavailable; treating current vote counts as if they existed at the historical cutoff would be leakage.

MovieLens timestamps are rating-event timestamps, not verified viewing timestamps. Accordingly, every output and manuscript statement must say “future positive ratings,” not “future watches.”

Protocol motivation: [Ji et al., data leakage in offline recommendation](https://arxiv.org/abs/2010.11060), [Gusak et al., global temporal splitting](https://arxiv.org/abs/2507.16289), [Krichene and Rendle, sampled metrics](https://dl.acm.org/doi/10.1145/3394486.3403226), and [Duricic et al., beyond-accuracy evaluation](https://arxiv.org/abs/2310.02294).


In [6]:
# ── Phase II configuration and environment ────────────────────────────────────
# Run All is intentional: split files are fixed and hashed before any model result is computed.
ENABLE_HEAVY_EVALUATION = True
INSTALL_DEPENDENCIES = True
AUTO_CLONE_RECBOLE_GNN = True
FORCE_RETRAIN = False
RUN_ITEMKNN = True
RUN_SEMANTIC = True

# Resource/quality controls. Reduce PRIMARY_USER_SAMPLE in Cell 1 only for debugging,
# never after inspecting method results. The publication run should use the declared 2,500.
EMBEDDING_SIZE = 512
N_LAYERS = 3
EPOCHS = 30
EARLY_STOPPING = 5
BPR_BATCH = 131072       # already measured at ~24 s/epoch; keep the proven setting
LIGHTGCN_BATCH = 524288  # fewer full-graph forward passes; comfortably uses more of a 40 GB A100
XSM_BATCH = 32768       # ~2x fewer graph propagations; safer than 32K for quadratic CL matrices
EVAL_BATCH = 65536       # evaluation-only batching; does not change rankings or metrics
SEM_MAX_POSITIVE_QUERIES = 24
SEM_MAX_NEGATIVE_QUERIES = 24
SEM_NEIGHBORS_PER_QUERY = 500
SEM_POOL = 2000
CF_POOL = 2000
TOPK_VALUES = (10, 20, 50)
OUTPUT_LIST_K = 100
BOOTSTRAP_REPLICATES = 5000

if not ENABLE_HEAVY_EVALUATION:
    raise RuntimeError('Heavy evaluation is disabled. Set ENABLE_HEAVY_EVALUATION=True when ready.')
if not WRITE_GLOBAL_SPLITS:
    raise RuntimeError('Set WRITE_GLOBAL_SPLITS=True in Cell 1 and rerun Cells 1–4 before training.')

import gc
import gzip
import math
import pickle
import platform
import random
import shutil
import subprocess
import sys
import time
from collections import defaultdict

if INSTALL_DEPENDENCIES:
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'recbole==1.1.1', 'faiss-gpu-cu12>=1.8.0,<2.0.0',
         'scipy>=1.11,<2', 'scikit-learn>=1.4,<2', 'pyyaml', 'tqdm'],
        check=True,
    )

# RecBole 1.1.1 compatibility with current NumPy/PyTorch.
for removed_name, replacement in {
    'float_': np.float64, 'complex_': np.complex128, 'unicode_': np.str_,
}.items():
    if not hasattr(np, removed_name):
        setattr(np, removed_name, replacement)
for alias, value in {
    'bool': np.bool_, 'int': np.int_, 'float': np.float64,
    'complex': np.complex128, 'object': np.object_, 'str': np.str_,
    'long': np.int_, 'unicode': np.str_,
}.items():
    if not hasattr(np, alias):
        setattr(np, alias, value)

import torch
if not hasattr(torch, '_cinematch_original_load'):
    torch._cinematch_original_load = torch.load
    def _safe_torch_load(path, *args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return torch._cinematch_original_load(path, *args, **kwargs)
    torch.load = _safe_torch_load

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = True

EVAL_OUT = OUT / 'publication_eval'
MODEL_OUT = EVAL_OUT / 'models'
EVAL_OUT.mkdir(parents=True, exist_ok=True)
MODEL_OUT.mkdir(parents=True, exist_ok=True)

gnn_candidates = [BASE / 'RecBole-GNN', Path('/content/RecBole-GNN')]
GNN_ROOT = next((path for path in gnn_candidates if (path / 'recbole_gnn').exists()), None)
if GNN_ROOT is None and AUTO_CLONE_RECBOLE_GNN:
    clone_target = Path('/content/RecBole-GNN')
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RUCAIBox/RecBole-GNN.git', str(clone_target)], check=True)
    GNN_ROOT = clone_target
if GNN_ROOT is None:
    raise FileNotFoundError('RecBole-GNN not found. Stage it under the project root or /content/RecBole-GNN.')
sys.path.insert(0, str(GNN_ROOT))

required_split_files = list(paths_to_write.values())
missing_split_files = [str(path) for path in required_split_files if not path.exists()]
assert not missing_split_files, f'Missing split files: {missing_split_files}'
print('Device:', DEVICE)
print('RecBole-GNN:', GNN_ROOT)
print('Evaluation output:', EVAL_OUT)


/tmp/ipykernel_2079/69321727.py:65: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias):
/tmp/ipykernel_2079/69321727.py:65: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias):


Device: cuda
RecBole-GNN: /content/RecBole-GNN
Evaluation output: /content/drive/MyDrive/cinematch/outputs/publication_audit/publication_eval


In [7]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)


Torch version: 2.11.0+cu128
CUDA available: True
CUDA version: 12.8


In [8]:
!pip install torch_geometric
!pip install pyg_lib torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.11.0+cu128.html

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.9 MB/s eta 0:00:00
Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 56.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 130.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 128.3 MB/s eta 0:00:00


In [9]:
# ── resume-safe RecBole/RecBole-GNN training ─────────────────────────
import scipy.sparse as sp
if hasattr(sp, 'dok_matrix') and not hasattr(sp.dok_matrix, '_update'):
    sp.dok_matrix._update = getattr(sp.dok_matrix, 'update', dict.update)
if hasattr(sp, 'dok_array') and not hasattr(sp.dok_array, '_update'):
    sp.dok_array._update = getattr(sp.dok_array, 'update', dict.update)

from recbole.utils import init_seed
from recbole_gnn.config import Config as GNNConfig
from recbole_gnn.utils import create_dataset, data_preparation, get_model, get_trainer


def release_gpu_memory(label=''):
    # Run between every model. Completed artifacts are already on Drive before cleanup.
    # IPython otherwise retains the previous exception traceback and its model tensors.
    for traceback_name in ('last_traceback', 'last_value', 'last_type'):
        if hasattr(sys, traceback_name):
            setattr(sys, traceback_name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except RuntimeError:
            pass
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        print(f'GPU cleanup {label}: {free_bytes / 2**30:.1f}/{total_bytes / 2**30:.1f} GiB free')


def _jsonable(value):
    if isinstance(value, dict):
        return {str(k): _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def _write_positive_inter(source_csv, inter_path):
    first = True
    for chunk in pd.read_csv(source_csv, chunksize=2_000_000, dtype=dtype):
        chunk = chunk.loc[chunk.rating >= 3.5, ['userId', 'movieId', 'rating', 'timestamp']].copy()
        chunk.columns = ['user_id:token', 'item_id:token', 'rating:float', 'timestamp:float']
        chunk.to_csv(inter_path, sep='\t', index=False, mode='w' if first else 'a', header=first)
        first = False
    assert inter_path.exists() and inter_path.stat().st_size > 0


def prepare_recbole_benchmark(train_csv, validation_csv, stage):
    data_root = MODEL_OUT / stage / 'dataset'
    dataset_name = f'cinematch_{stage}'
    dataset_dir = data_root / dataset_name
    train_path = dataset_dir / f'{dataset_name}.train.inter'
    valid_path = dataset_dir / f'{dataset_name}.valid.inter'
    test_path = dataset_dir / f'{dataset_name}.test.inter'
    done_path = dataset_dir / 'benchmark_fingerprint.json'
    train_key = 'development_train' if stage == 'development' else 'final_train'
    fingerprint = {
        'train_sha256': manifest.get('sha256', {}).get(train_key),
        'validation_sha256': manifest.get('sha256', {}).get('validation_targets'),
    }
    if all(path.exists() for path in [train_path, valid_path, test_path]) and done_path.exists():
        if json.loads(done_path.read_text()) == fingerprint:
            return data_root, dataset_name, fingerprint
    dataset_dir.mkdir(parents=True, exist_ok=True)
    _write_positive_inter(train_csv, train_path)
    _write_positive_inter(validation_csv, valid_path)
    shutil.copyfile(valid_path, test_path)  # Required by RecBole; final training never evaluates it.
    done_path.write_text(json.dumps(fingerprint, sort_keys=True))
    return data_root, dataset_name, fingerprint


def _token_map(dataset, field):
    tokens = dataset.field2id_token[field]
    result = {}
    for row, token in enumerate(tokens):
        token = str(token)
        if token == '[PAD]':
            continue
        try:
            result[int(float(token))] = int(row)
        except ValueError:
            continue
    return result


def train_and_export(model_name, stage, source_csv, validation_csv, fixed_epochs=None):
    tag = model_name.lower()
    export_dir = MODEL_OUT / stage / tag
    done_file = export_dir / 'done.json'
    requested_epochs = int(fixed_epochs or EPOCHS)
    data_root, dataset_name, source_fingerprint = prepare_recbole_benchmark(
        source_csv, validation_csv, stage
    )
    if done_file.exists() and not FORCE_RETRAIN:
        completed = json.loads(done_file.read_text())
        completed_hparams = completed.get('effective_hyperparameters', {})
        graph_config_ok = (model_name == 'BPR' or completed_hparams.get('enable_sparse') is True)
        if (completed.get('source_fingerprint') == source_fingerprint
                and completed.get('requested_epochs') == requested_epochs
                and graph_config_ok):
            print(f'Loading completed {stage}/{model_name}')
            return completed
        print(f'Ignoring stale {stage}/{model_name}: data or epoch protocol changed')

    checkpoint_dir = export_dir / 'checkpoints'
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    batch_size = {
        'BPR': BPR_BATCH,
        'LightGCN': LIGHTGCN_BATCH,
        'XSimGCL': XSM_BATCH,
    }[model_name]
    config_dict = {
        'model': model_name,
        'dataset': dataset_name,
        'data_path': str(data_root),
        'USER_ID_FIELD': 'user_id', 'ITEM_ID_FIELD': 'item_id',
        'RATING_FIELD': 'rating', 'TIME_FIELD': 'timestamp',
        'load_col': {'inter': ['user_id', 'item_id', 'rating', 'timestamp']},
        'threshold': {'rating': 3.5},
        'user_inter_num_interval': '[1,inf)', 'item_inter_num_interval': '[1,inf)',
        # Exact pre-split files: development uses the declared validation window;
        # final fits every pre-test interaction for the epoch count selected in development.
        'benchmark_filename': ['train', 'valid', 'test'],
        'eval_args': {'split': {'RS': [1, 0, 0]}, 'group_by': 'user', 'order': 'TO', 'mode': 'uni100'},
        'metrics': ['Recall', 'NDCG', 'MRR'], 'topk': [10, 20, 50],
        'valid_metric': 'NDCG@20', 'valid_metric_bigger': True,
        'embedding_size': EMBEDDING_SIZE, 'n_layers': N_LAYERS,
        # Critical at MovieLens-32M scale: PyG's dense edge-message route attempts ~60 GiB at 512d.
        # SparseTensor propagation is mathematically the same normalized LightGCN aggregation.
        'enable_sparse': model_name in {'LightGCN', 'XSimGCL'},
        'reg_weight': 1e-4, 'learning_rate': 1e-3,
        'train_batch_size': batch_size, 'eval_batch_size': EVAL_BATCH,
        'epochs': requested_epochs,
        'eval_step': 3 if model_name == 'XSimGCL' else 1,
        'stopping_step': EARLY_STOPPING,
        'seed': SEED, 'reproducibility': True, 'show_progress': True,
        'save_dataset': False, 'save_dataloaders': False, 'checkpoint_dir': str(checkpoint_dir),
        'use_gpu': torch.cuda.is_available(), 'gpu_id': 0 if torch.cuda.is_available() else -1,
        # Effective XSimGCL parameter names from RecBole-GNN, not ignored aliases.
        'lambda': 0.1, 'eps': 0.2, 'temperature': 0.2, 'layer_cl': 1, 'require_pow': True,
    }
    config = GNNConfig(model=model_name, dataset=dataset_name, config_dict=config_dict)
    init_seed(config['seed'], config['reproducibility'])
    dataset = create_dataset(config)
    train_data, valid_data, test_data = data_preparation(config, dataset)
    model_class = get_model(model_name)
    model = model_class(config, train_data.dataset).to(config['device'])
    if model_name in {'LightGCN', 'XSimGCL'}:
        assert getattr(model, 'use_sparse', False), (
            'Sparse graph propagation is unavailable. Install a torch-sparse build matching the current '
            'PyTorch/CUDA versions; dense propagation cannot fit MovieLens-32M at 512 dimensions.'
        )
    trainer_class = get_trainer(config['MODEL_TYPE'], model_name)
    trainer = trainer_class(config, model)
    started = time.time()
    fit_validation = valid_data if stage == 'development' else None
    best_valid_score, best_valid_result = trainer.fit(
        train_data, fit_validation, saved=True, show_progress=True
    )
    checkpoint = torch.load(trainer.saved_model_file, map_location='cpu')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    with torch.no_grad():
        if model_name in {'LightGCN', 'XSimGCL'}:
            propagated = model.forward()
            user_emb, item_emb = propagated[0], propagated[1]
        else:
            user_emb, item_emb = model.user_embedding.weight, model.item_embedding.weight
        user_emb = user_emb.detach().cpu().float().numpy()
        item_emb = item_emb.detach().cpu().float().numpy()
    export_dir.mkdir(parents=True, exist_ok=True)
    np.save(export_dir / 'user_emb.npy', user_emb)
    np.save(export_dir / 'item_emb.npy', item_emb)
    user_map = _token_map(dataset, dataset.uid_field)
    item_map = _token_map(dataset, dataset.iid_field)
    (export_dir / 'user_map.json').write_text(json.dumps({str(k): v for k, v in user_map.items()}))
    (export_dir / 'item_map.json').write_text(json.dumps({str(k): v for k, v in item_map.items()}))
    result = {
        'model': model_name, 'stage': stage, 'source_csv': str(source_csv),
        'source_fingerprint': source_fingerprint,
        'requested_epochs': requested_epochs,
        'selected_checkpoint_epoch_zero_based': int(checkpoint.get('epoch', requested_epochs - 1)),
        'training_protocol': ('development early stopping on declared validation window'
                              if stage == 'development' else 'full pre-test refit with development-selected epoch count'),
        'best_valid_score': float(best_valid_score), 'best_valid_result': _jsonable(best_valid_result),
        'elapsed_seconds': float(time.time() - started),
        'user_embedding_shape': list(user_emb.shape), 'item_embedding_shape': list(item_emb.shape),
        'effective_hyperparameters': {k: _jsonable(config_dict[k]) for k in [
            'embedding_size', 'n_layers', 'reg_weight', 'learning_rate', 'train_batch_size',
            'epochs', 'enable_sparse', 'lambda', 'eps', 'temperature', 'layer_cl', 'seed']},
    }
    done_file.write_text(json.dumps(result, indent=2))
    if model_name in {'LightGCN', 'XSimGCL'}:
        del propagated
    del model, trainer, train_data, valid_data, test_data, dataset, checkpoint, user_emb, item_emb
    release_gpu_memory(f'after {stage}/{model_name}')
    print(f'Completed {stage}/{model_name}: {result["elapsed_seconds"] / 60:.1f} min')
    return result


training_results_path = EVAL_OUT / 'training_results.json'
training_results = json.loads(training_results_path.read_text()) if training_results_path.exists() else {}
for _model_name in ['BPR', 'LightGCN', 'XSimGCL']:
    release_gpu_memory(f'before development/{_model_name}')
    development_result = train_and_export(
        _model_name, 'development', paths_to_write['development_train'], paths_to_write['validation_targets']
    )
    training_results[f'development_{_model_name.lower()}'] = development_result
    training_results_path.write_text(json.dumps(training_results, indent=2))
    release_gpu_memory(f'before final/{_model_name}')
    selected_epochs = development_result['selected_checkpoint_epoch_zero_based'] + 1
    training_results[f'final_{_model_name.lower()}'] = train_and_export(
        _model_name, 'final', paths_to_write['final_train'], paths_to_write['validation_targets'],
        fixed_epochs=selected_epochs,
    )
    training_results_path.write_text(json.dumps(training_results, indent=2))
    release_gpu_memory(f'completed {_model_name}')


GPU cleanup before development/BPR: 39.1/39.5 GiB free
Loading completed development/BPR
GPU cleanup before final/BPR: 39.1/39.5 GiB free
Loading completed final/BPR
GPU cleanup completed BPR: 39.1/39.5 GiB free
GPU cleanup before development/LightGCN: 39.1/39.5 GiB free
Loading completed development/LightGCN
GPU cleanup before final/LightGCN: 39.1/39.5 GiB free
Loading completed final/LightGCN


GPU cleanup completed LightGCN: 39.1/39.5 GiB free
GPU cleanup before development/XSimGCL: 39.1/39.5 GiB free


/usr/local/lib/python3.13/dist-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/usr/local/lib/python3.13/dist-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

GPU cleanup after development/XSimGCL: 39.0/39.5 GiB free
Completed development/XSimGCL: 184.1 min


GPU cleanup before final/XSimGCL: 39.0/39.5 GiB free


/usr/local/lib/python3.13/dist-packages/recbole/data/dataset/dataset.py:638: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
/usr/local/lib/python3.13/dist-packages/recbole/data/dataset/dataset.py:640: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a c

GPU cleanup after final/XSimGCL: 39.0/39.5 GiB free
Completed final/XSimGCL: 236.2 min
GPU cleanup completed XSimGCL: 39.0/39.5 GiB free


In [7]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [8]:
# ── Metadata, production-faithful features, and semantic RRF/Rocchio ───────────
movies = pd.read_csv(BASE / 'Data/ml-32m/movies.csv')
movie_genres = {
    int(mid): set(str(genres).split('|')) - {'(no genres listed)'}
    for mid, genres in zip(movies.movieId, movies.genres)
}

catalog_columns = pd.read_csv(CATALOG, nrows=0).columns.tolist()
needed_catalog_columns = [column for column in [
    'id', 'imdb_id', 'original_language', 'imdb_rating', 'imdb_votes',
    'release_date', 'year', 'genres', 'title'
] if column in catalog_columns]
catalog_full = pd.read_csv(CATALOG, usecols=needed_catalog_columns, low_memory=False)
catalog_full['id'] = pd.to_numeric(catalog_full['id'], errors='coerce')
catalog_full = catalog_full.dropna(subset=['id']).drop_duplicates('id')
links_full = pd.read_csv(LINKS, usecols=['movieId', 'tmdbId'])
links_full['tmdbId'] = pd.to_numeric(links_full.tmdbId, errors='coerce')
movie_meta = links_full.merge(catalog_full, left_on='tmdbId', right_on='id', how='left')
movie_meta['movieId'] = movie_meta.movieId.astype(int)
movie_meta = movie_meta.set_index('movieId', drop=False)

def _numeric_meta(column, default=0.0):
    if column not in movie_meta.columns:
        return pd.Series(default, index=movie_meta.index, dtype=float)
    return pd.to_numeric(movie_meta[column], errors='coerce').fillna(default).astype(float)

_rating = _numeric_meta('imdb_rating', 6.5)
_votes = _numeric_meta('imdb_votes', 0.0).clip(lower=0)
_language = movie_meta.get('original_language', pd.Series(index=movie_meta.index, dtype='object')).astype('string').str.lower()
_bayes_m = _language.map({
    'en': 10000, 'hi': 3000, 'te': 2000, 'ta': 2000, 'ml': 1500, 'kn': 1500,
    'bn': 1500, 'mr': 1500, 'ko': 4000, 'ja': 4000, 'zh': 3000, 'cn': 3000,
    'fr': 3000, 'es': 3000, 'de': 3000, 'pt': 2000, 'it': 2000,
}).fillna(2000).astype(float)
movie_meta['quality_bayes'] = (_votes / (_votes + _bayes_m)) * _rating + (_bayes_m / (_votes + _bayes_m)) * 6.5
movie_meta['quality_n'] = movie_meta.groupby(_language, dropna=False)['quality_bayes'].transform(
    lambda values: (values - values.min()) / (values.max() - values.min()) if values.max() > values.min() else 0.5
).fillna(0.5)
_non_en = _language.notna() & _language.ne('en')
_strong = (_votes >= 5000) & (_rating >= 6.4)
_sparse_ok = movie_meta.quality_bayes >= 7.0
movie_meta['passes_imdb_snapshot_gate'] = (~_non_en) | _strong | _sparse_ok

quality_score = movie_meta.quality_n.to_dict()
quality_gate = movie_meta.passes_imdb_snapshot_gate.to_dict()
language_map = {int(mid): (None if pd.isna(lang) or not str(lang).strip() else str(lang).strip().lower())
                for mid, lang in _language.items()}

IMDB_INDEX_CANDIDATES = [
    BASE / 'models/imdbbge/imdb_movies_bge_m3_flatip.faiss',
    BASE / 'outputs/imdb/imdb_movies_bge_m3_flatip.faiss',
    BASE / 'Data/outputs/imdb/imdb_movies_bge_m3_flatip.faiss',
]
IMDB_META_CANDIDATES = [
    BASE / 'models/imdbbge/imdb_movies_meta.csv',
    BASE / 'outputs/imdb/imdb_movies_meta.csv',
    BASE / 'Data/outputs/imdb/imdb_movies_meta.csv',
]
IMDB_INDEX_PATH = next((path for path in IMDB_INDEX_CANDIDATES if path.exists()), None)
IMDB_META_PATH = next((path for path in IMDB_META_CANDIDATES if path.exists()), None)
if RUN_SEMANTIC:
    assert IMDB_INDEX_PATH is not None, 'IMDb BGE-M3 FAISS index not found.'
    assert IMDB_META_PATH is not None, 'IMDb BGE-M3 metadata not found.'
    import faiss
    semantic_meta = pd.read_csv(IMDB_META_PATH, usecols=['row_id', 'tconst'])
    row_to_tconst = dict(zip(semantic_meta.row_id.astype(int), semantic_meta.tconst.astype(str)))
    tconst_to_row = {tconst: row for row, tconst in row_to_tconst.items()}
    tmdb_to_tconst = {int(row.id): str(row.imdb_id) for row in catalog_full[['id', 'imdb_id']].dropna().itertuples(index=False)}
    tconst_to_tmdb = {value: key for key, value in tmdb_to_tconst.items()}
    movie_to_semantic_row = {}
    for movie_id, tmdb_id in ml_to_tmdb.items():
        tconst = tmdb_to_tconst.get(int(tmdb_id))
        row = tconst_to_row.get(tconst)
        if row is not None:
            movie_to_semantic_row[int(movie_id)] = int(row)
    _cpu_index = faiss.read_index(str(IMDB_INDEX_PATH))
    try:
        _gpu_resources = faiss.StandardGpuResources()
        SEMANTIC_INDEX = faiss.index_cpu_to_gpu(_gpu_resources, 0, _cpu_index)
        print('Semantic FAISS index on GPU:', SEMANTIC_INDEX.ntotal)
    except Exception as error:
        SEMANTIC_INDEX = _cpu_index
        print('Semantic FAISS CPU fallback:', str(error)[:120])


def semantic_rrf(history_ratings, allowed_candidates, use_rocchio=False):
    if not RUN_SEMANTIC:
        return {}
    ordered = sorted(history_ratings.items(), key=lambda pair: (pair[1][1], pair[0]))
    positives = [mid for mid, (rating, _) in ordered if rating >= 3.5 and mid in movie_to_semantic_row][-SEM_MAX_POSITIVE_QUERIES:]
    negatives = [mid for mid, (rating, _) in ordered if rating <= 2.5 and mid in movie_to_semantic_row][-SEM_MAX_NEGATIVE_QUERIES:]
    if not positives:
        return {}
    # Reconstruct on CPU: GPU FAISS wrappers do not consistently expose reconstruct().
    queries = np.vstack([_cpu_index.reconstruct(int(movie_to_semantic_row[mid])) for mid in positives]).astype('float32')
    if use_rocchio and negatives:
        negative_vectors = np.vstack([_cpu_index.reconstruct(int(movie_to_semantic_row[mid])) for mid in negatives]).astype('float32')
        queries = queries - 0.45 * negative_vectors.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(queries, axis=1, keepdims=True); norms[norms == 0] = 1.0
    queries /= norms
    _, neighbor_rows = SEMANTIC_INDEX.search(queries, SEM_NEIGHBORS_PER_QUERY)
    seen = set(history_ratings)
    allowed = set(map(int, allowed_candidates))
    scores = defaultdict(float)
    for neighbors in neighbor_rows:
        for rank, semantic_row in enumerate(neighbors, start=1):
            tconst = row_to_tconst.get(int(semantic_row))
            tmdb_id = tconst_to_tmdb.get(tconst)
            movie_id = tmdb_to_ml.get(int(tmdb_id)) if tmdb_id is not None else None
            if movie_id is not None and movie_id in allowed and movie_id not in seen:
                scores[int(movie_id)] += 1.0 / (60.0 + rank)
    if not scores:
        return {}
    maximum = max(scores.values())
    return dict(sorted(((mid, score / maximum) for mid, score in scores.items()), key=lambda pair: pair[1], reverse=True)[:SEM_POOL])

print('Movie metadata coverage:', len(movie_meta), 'MovieLens rows; language mapped:', sum(v is not None for v in language_map.values()))


Semantic FAISS index on GPU: 737654
Movie metadata coverage: 87585 MovieLens rows; language mapped: 80387


In [9]:
# ── Stage contexts, collaborative scores, ItemKNN, and shared metrics ───────────
def load_embedding_bundle(stage, model_name):
    root = MODEL_OUT / stage / model_name.lower()
    required = [root / 'user_emb.npy', root / 'item_emb.npy', root / 'user_map.json', root / 'item_map.json']
    assert all(path.exists() for path in required), f'Missing {stage}/{model_name} artifacts.'
    return {
        'user_emb': np.load(required[0], mmap_mode='r'),
        'item_emb': np.load(required[1], mmap_mode='r'),
        'user_map': {int(k): int(v) for k, v in json.loads(required[2].read_text()).items()},
        'item_map': {int(k): int(v) for k, v in json.loads(required[3].read_text()).items()},
    }


def build_context(stage, user_ids, targets_frame, candidate_ids):
    mask = dev_mask if stage == 'development' else final_train_mask
    user_ids = sorted(set(map(int, user_ids)))
    selected_rows = ratings.loc[mask & ratings.userId.isin(user_ids), ['userId', 'movieId', 'rating', 'timestamp']]
    histories = {uid: {} for uid in user_ids}
    for row in selected_rows.itertuples(index=False):
        histories[int(row.userId)][int(row.movieId)] = (float(row.rating), int(row.timestamp))
    positive_rows = ratings.loc[mask & ratings.rating.ge(3.5), ['movieId']]
    popularity = positive_rows.groupby('movieId').size().astype(int).to_dict()
    candidate_ids = sorted(set(map(int, candidate_ids)))
    popularity_order = sorted(candidate_ids, key=lambda mid: (-popularity.get(mid, 0), mid))
    pop_values = np.array([popularity.get(mid, 0) for mid in candidate_ids], dtype=float)
    tail_threshold = float(np.quantile(pop_values, 0.80)) if len(pop_values) else 0.0
    ground_truth = targets_frame.groupby('userId').movieId.apply(lambda values: set(map(int, values))).to_dict()
    return {
        'stage': stage, 'users': user_ids, 'histories': histories, 'popularity': popularity,
        'popularity_order': popularity_order, 'candidate_ids': candidate_ids,
        'candidate_set': set(candidate_ids), 'tail_threshold': tail_threshold,
        'ground_truth': ground_truth,
    }


def collaborative_outputs(bundle, context, pool_size=CF_POOL, list_k=OUTPUT_LIST_K, batch_size=128):
    candidate_pairs = [(mid, bundle['item_map'][mid]) for mid in context['candidate_ids'] if mid in bundle['item_map']]
    candidate_mids = np.array([pair[0] for pair in candidate_pairs], dtype=np.int64)
    candidate_rows = np.array([pair[1] for pair in candidate_pairs], dtype=np.int64)
    candidate_position = {int(mid): idx for idx, mid in enumerate(candidate_mids)}
    item_tensor = torch.as_tensor(np.asarray(bundle['item_emb'][candidate_rows]), device=DEVICE)
    all_scores, all_recommendations = {}, {}
    users = context['users']
    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        valid = [(uid, bundle['user_map'].get(uid)) for uid in batch_users]
        valid = [(uid, row) for uid, row in valid if row is not None and row < len(bundle['user_emb'])]
        if not valid:
            continue
        user_tensor = torch.as_tensor(np.asarray(bundle['user_emb'][[row for _, row in valid]]), device=DEVICE)
        scores = user_tensor @ item_tensor.T
        for batch_row, (uid, _) in enumerate(valid):
            for seen_mid in context['histories'][uid]:
                position = candidate_position.get(int(seen_mid))
                if position is not None:
                    scores[batch_row, position] = -torch.inf
        take = min(pool_size, scores.shape[1])
        values, positions = torch.topk(scores, k=take, dim=1)
        values = values.detach().cpu().numpy(); positions = positions.detach().cpu().numpy()
        for row_index, (uid, _) in enumerate(valid):
            mids = candidate_mids[positions[row_index]]
            vals = values[row_index]
            finite = np.isfinite(vals)
            mids, vals = mids[finite], vals[finite]
            all_scores[uid] = {int(mid): float(score) for mid, score in zip(mids, vals)}
            all_recommendations[uid] = [int(mid) for mid in mids[:list_k]]
        del scores, user_tensor, values, positions
    del item_tensor
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return all_scores, all_recommendations


def build_itemknn_matrix(mask):
    positives = ratings.loc[mask & ratings.rating.ge(3.5), ['userId', 'movieId']]
    items = np.sort(positives.movieId.unique())
    users = np.sort(positives.userId.unique())
    item_to_row = {int(mid): idx for idx, mid in enumerate(items)}
    user_to_col = {int(uid): idx for idx, uid in enumerate(users)}
    rows = positives.movieId.map(item_to_row).to_numpy()
    cols = positives.userId.map(user_to_col).to_numpy()
    matrix = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(items), len(users))).tocsr()
    norms = np.sqrt(matrix.multiply(matrix).sum(axis=1)).A1; norms[norms == 0] = 1.0
    return matrix.multiply(1.0 / norms[:, None]).tocsr(), items, item_to_row


def build_itemknn_recommendations(context, cache_name='test_itemknn', k=OUTPUT_LIST_K, batch_size=32):
    # Exact same score as sum_i cosine(i, candidate), but use associativity:
    # (batched liked-item profiles @ normalized item-user matrix) @ matrix.T.
    # This avoids materializing up to 50 separate item-item rows per user.
    signature_payload = {
        'users': list(map(int, context['users'])),
        'candidates': list(map(int, context['candidate_ids'])),
        'history_sha256': hashlib.sha256(json.dumps({
            str(uid): sorted(map(int, context['histories'][uid])) for uid in context['users']
        }, sort_keys=True).encode()).hexdigest(),
        'k': int(k), 'history_limit': 50, 'positive_threshold': 3.5,
    }
    signature = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode()).hexdigest()[:16]
    cache_path = EVAL_OUT / f'{cache_name}_{signature}.pkl.gz'
    recommendations = {}
    if cache_path.exists():
        with gzip.open(cache_path, 'rb') as stream:
            saved = pickle.load(stream)
        if saved.get('signature') == signature:
            recommendations = {int(uid): list(map(int, recs)) for uid, recs in saved['recommendations'].items()}
            if len(recommendations) == len(context['users']):
                print(f'Loaded completed ItemKNN cache: {len(recommendations)} users')
                return recommendations
            print(f'Resuming ItemKNN cache: {len(recommendations)}/{len(context["users"])} users')

    matrix, items, item_to_row = build_itemknn_matrix(final_train_mask)
    candidate_rows = np.array([item_to_row[mid] for mid in context['candidate_ids'] if mid in item_to_row], dtype=np.int64)
    candidate_mask = np.zeros(len(items), dtype=bool)
    candidate_mask[candidate_rows] = True
    pending = [uid for uid in context['users'] if uid not in recommendations]
    for start in range(0, len(pending), batch_size):
        batch_users = pending[start:start + batch_size]
        profile_rows, profile_cols = [], []
        users_with_likes = []
        for local_row, uid in enumerate(batch_users):
            ordered = sorted(context['histories'][uid].items(), key=lambda pair: (pair[1][1], pair[0]))
            liked = [item_to_row[mid] for mid, (rating, _) in ordered if rating >= 3.5 and mid in item_to_row][-50:]
            if liked:
                profile_rows.extend([local_row] * len(liked)); profile_cols.extend(liked)
                users_with_likes.append(local_row)
            else:
                recommendations[int(uid)] = [mid for mid in context['popularity_order'] if mid not in context['histories'][uid]][:k]
        if profile_rows:
            profiles = sp.csr_matrix(
                (np.ones(len(profile_rows), dtype=np.float32), (profile_rows, profile_cols)),
                shape=(len(batch_users), matrix.shape[0]),
            )
            scores = ((profiles @ matrix) @ matrix.T).tocsr()
            for local_row in users_with_likes:
                uid = int(batch_users[local_row])
                similarity = scores.getrow(local_row).toarray().ravel()
                valid = candidate_mask.copy()
                for seen_mid in context['histories'][uid]:
                    seen_row = item_to_row.get(int(seen_mid))
                    if seen_row is not None: valid[seen_row] = False
                similarity[~valid] = -np.inf
                positive_count = int(np.count_nonzero(similarity > 0))
                take = min(k, positive_count)
                if take:
                    order = np.argpartition(-similarity, take - 1)[:take]
                    order = order[np.argsort(-similarity[order], kind='stable')]
                    recommendations[uid] = items[order].astype(int).tolist()
                else:
                    recommendations[uid] = []
            del profiles, scores
        with gzip.open(cache_path, 'wb') as stream:
            pickle.dump({'signature': signature, 'recommendations': recommendations}, stream, protocol=pickle.HIGHEST_PROTOCOL)
        print(f'ItemKNN: {len(recommendations)}/{len(context["users"])} users checkpointed')
    del matrix
    gc.collect()
    return recommendations


def ndcg_at_k(recommended, truth, k):
    dcg = sum(1.0 / math.log2(rank + 2) for rank, mid in enumerate(recommended[:k]) if mid in truth)
    ideal = sum(1.0 / math.log2(rank + 2) for rank in range(min(len(truth), k)))
    return dcg / ideal if ideal else 0.0


def recall_at_k(recommended, truth, k):
    return len(set(recommended[:k]) & truth) / len(truth) if truth else 0.0


def precision_at_k(recommended, truth, k):
    return len(set(recommended[:k]) & truth) / k if k else 0.0


def hit_at_k(recommended, truth, k):
    return float(bool(set(recommended[:k]) & truth))


def mrr_at_k(recommended, truth, k):
    return next((1.0 / (rank + 1) for rank, mid in enumerate(recommended[:k]) if mid in truth), 0.0)


def average_precision_at_k(recommended, truth, k):
    hits = 0; total = 0.0
    for rank, mid in enumerate(recommended[:k], start=1):
        if mid in truth:
            hits += 1; total += hits / rank
    return total / min(len(truth), k) if truth else 0.0


def ild_at_k(recommended, k):
    lists = [movie_genres.get(int(mid), set()) for mid in recommended[:k]]
    if len(lists) < 2: return 0.0
    distances = []
    for left in range(len(lists)):
        for right in range(left + 1, len(lists)):
            union = lists[left] | lists[right]
            distances.append(1.0 - len(lists[left] & lists[right]) / len(union) if union else 0.0)
    return float(np.mean(distances))


def history_dominant_language(history_ratings):
    languages = [language_map.get(mid) for mid, (rating, _) in history_ratings.items() if rating >= POSITIVE_THRESHOLD and language_map.get(mid)]
    if len(languages) < MIN_KNOWN_HISTORY_POSITIVES: return None
    counts = Counter(languages); top = max(counts.values())
    leaders = [lang for lang, count in counts.items() if count == top]
    return leaders[0] if len(leaders) == 1 and top / len(languages) >= MIN_DOMINANT_LANGUAGE_SHARE else None


def evaluate_recommendations(method_recommendations, context, cohort_name):
    per_user_rows = []
    aggregate_rows = []
    total_popularity = sum(context['popularity'].values()) + len(context['candidate_ids'])
    for method, user_recommendations in method_recommendations.items():
        recommended_catalog = set()
        for uid, truth in context['ground_truth'].items():
            rec = list(dict.fromkeys(user_recommendations.get(uid, [])))[:OUTPUT_LIST_K]
            if not rec: continue
            recommended_catalog.update(rec)
            dominant = history_dominant_language(context['histories'][uid])
            mapped = [language_map.get(mid) for mid in rec[:10] if language_map.get(mid)]
            non_dominant = (sum(lang != dominant for lang in mapped) / len(mapped)) if mapped and dominant else np.nan
            novelty = np.mean([-math.log2((context['popularity'].get(mid, 0) + 1) / total_popularity) for mid in rec[:10]])
            tail_share = np.mean([context['popularity'].get(mid, 0) <= context['tail_threshold'] for mid in rec[:10]])
            row = {'cohort': cohort_name, 'method': method, 'userId': uid, 'targets': len(truth),
                   'ILD@10': ild_at_k(rec, 10), 'Novelty@10': float(novelty), 'LongTail@10': float(tail_share),
                   'LanguageMapCoverage@10': len(mapped) / min(len(rec), 10),
                   'NonDominantLanguageExposure@10': non_dominant,
                   'MeanCurrentIMDbQuality@10': float(np.mean([quality_score.get(mid, 0.5) for mid in rec[:10]]))}
            for k in TOPK_VALUES:
                row[f'NDCG@{k}'] = ndcg_at_k(rec, truth, k)
                row[f'Recall@{k}'] = recall_at_k(rec, truth, k)
                row[f'Precision@{k}'] = precision_at_k(rec, truth, k)
                row[f'Hit@{k}'] = hit_at_k(rec, truth, k)
                row[f'MRR@{k}'] = mrr_at_k(rec, truth, k)
                row[f'MAP@{k}'] = average_precision_at_k(rec, truth, k)
            per_user_rows.append(row)
        method_rows = [row for row in per_user_rows if row['method'] == method and row['cohort'] == cohort_name]
        if not method_rows: continue
        frame = pd.DataFrame(method_rows)
        aggregate = {'cohort': cohort_name, 'method': method, 'n_users': len(frame),
                     'CatalogCoverage': len(recommended_catalog) / max(len(context['candidate_ids']), 1)}
        for column in frame.columns:
            if column not in {'cohort', 'method', 'userId'} and pd.api.types.is_numeric_dtype(frame[column]):
                aggregate[column] = float(frame[column].mean())
        aggregate_rows.append(aggregate)
    return pd.DataFrame(aggregate_rows), pd.DataFrame(per_user_rows)


In [10]:
# ── Signal caching, fair fusion, production ablations, and DPP ─────────────────
def minmax_scores(scores, candidates):
    candidates = list(candidates)
    if not candidates: return {}
    values = np.array([scores.get(mid, 0.0) for mid in candidates], dtype=float)
    low, high = float(values.min()), float(values.max())
    scale = high - low if high > low else 1.0
    return {mid: (scores.get(mid, 0.0) - low) / scale for mid in candidates}


def build_semantic_cache(context, cache_name, rocchio=False):
    signature_payload = {
        'users': list(map(int, context['users'])),
        'candidates': list(map(int, context['candidate_ids'])),
        'targets': {str(uid): sorted(map(int, truth)) for uid, truth in context['ground_truth'].items()},
        'rocchio': bool(rocchio),
    }
    signature = hashlib.sha256(json.dumps(signature_payload, sort_keys=True).encode()).hexdigest()[:16]
    cache_path = EVAL_OUT / f'{cache_name}_{signature}.pkl.gz'
    if cache_path.exists() and not FORCE_RETRAIN:
        with gzip.open(cache_path, 'rb') as stream: return pickle.load(stream)
    cache = {}
    for index, uid in enumerate(context['users']):
        cache[uid] = semantic_rrf(context['histories'][uid], context['candidate_set'], use_rocchio=rocchio)
        if index % 100 == 0: print(f'{cache_name}: {index}/{len(context["users"])}')
    with gzip.open(cache_path, 'wb') as stream: pickle.dump(cache, stream, protocol=pickle.HIGHEST_PROTOCOL)
    return cache


def liked_language_fit(uid, mid, context):
    languages = [language_map.get(item) for item, (rating, _) in context['histories'][uid].items()
                 if rating >= 3.5 and language_map.get(item)]
    language = language_map.get(mid)
    if not language: return 0.15
    counts = Counter(languages)
    if language in counts:
        return min(1.0, 0.65 + 0.30 * counts[language] / max(counts.values()))
    dominant = history_dominant_language(context['histories'][uid])
    if language == 'en': return 0.50 if dominant == 'en' else 0.25
    return 0.05 if dominant == 'en' else 0.15


def liked_genre_fit(uid, mid, context):
    counts = Counter()
    for item, (rating, _) in context['histories'][uid].items():
        if rating >= 3.5: counts.update(movie_genres.get(item, set()))
    genres = movie_genres.get(mid, set())
    if not counts or not genres: return 0.0
    return max(counts.get(genre, 0) for genre in genres) / max(counts.values())


def fused_scores(uid, context, semantic_scores, cf_scores, alpha, variant):
    seen = set(context['histories'][uid])
    candidates = (set(semantic_scores) | set(cf_scores)) - seen
    if not candidates:
        return {mid: float(len(context['popularity_order']) - rank) for rank, mid in enumerate(context['popularity_order'][:CF_POOL]) if mid not in seen}
    semantic_n = minmax_scores(semantic_scores, candidates)
    cf_n = minmax_scores(cf_scores, candidates)
    beta = 1.0 - alpha
    base = {mid: alpha * semantic_n[mid] + beta * cf_n[mid] for mid in candidates}
    if variant == 'FusionFrozen': return base
    semantic_values = np.array(list(semantic_n.values()))
    bridge_threshold = float(np.quantile(semantic_values, 0.99)) if len(semantic_values) else 1.0
    warm = sum(rating >= 3.5 for rating, _ in context['histories'][uid].values()) >= 15
    if warm:
        weights = {'sem': 0.14, 'cf': 0.48, 'quality': 0.13, 'language': 0.11, 'genre': 0.07}
    else:
        weights = {'sem': 0.34, 'cf': 0.23, 'quality': 0.22, 'language': 0.09, 'genre': 0.05}
    dominant = history_dominant_language(context['histories'][uid])
    output = {}
    for mid in candidates:
        language_fit = liked_language_fit(uid, mid, context)
        candidate_language = language_map.get(mid)
        bridge = 0.40 if (candidate_language and dominant and candidate_language != dominant and semantic_n[mid] >= bridge_threshold) else 0.0
        if variant == 'FusionLanguageBridge':
            output[mid] = base[mid] + 0.10 * language_fit + bridge
        else:
            output[mid] = (weights['sem'] * semantic_n[mid] + weights['cf'] * cf_n[mid]
                           + weights['quality'] * quality_score.get(mid, 0.5)
                           + weights['language'] * language_fit
                           + weights['genre'] * liked_genre_fit(uid, mid, context) + bridge)
    return output


def greedy_dpp(ranked_mids, score_map, k=OUTPUT_LIST_K, topn=200, language_weight=0.4):
    candidates = ranked_mids[:topn]
    if len(candidates) <= 1: return candidates
    genre_vocab = {genre: idx for idx, genre in enumerate(sorted({g for mid in candidates for g in movie_genres.get(mid, set())}))}
    language_vocab = {lang: idx for idx, lang in enumerate(sorted({language_map.get(mid) for mid in candidates if language_map.get(mid)}))}
    if not genre_vocab and not language_vocab:
        return candidates[:k]
    features = np.zeros((len(candidates), len(genre_vocab) + len(language_vocab)), dtype=np.float64)
    for row, mid in enumerate(candidates):
        for genre in movie_genres.get(mid, set()):
            if genre in genre_vocab: features[row, genre_vocab[genre]] = 0.6
        lang = language_map.get(mid)
        if lang in language_vocab: features[row, len(genre_vocab) + language_vocab[lang]] = language_weight
    norms = np.linalg.norm(features, axis=1, keepdims=True); norms[norms == 0] = 1.0
    features /= norms
    similarity = features @ features.T
    quality = np.array([max(score_map.get(mid, 0.0), 1e-4) for mid in candidates])
    quality /= max(quality.max(), 1e-8)
    kernel = np.outer(quality, quality) * similarity
    chosen = []
    cis = np.zeros((len(candidates), 0), dtype=np.float64)
    gains = np.diag(kernel).copy()
    for _ in range(min(k, len(candidates))):
        index = int(np.argmax(gains))
        if gains[index] < 0: break
        chosen.append(index)
        denominator = math.sqrt(max(gains[index], 1e-12))
        update = (kernel[:, index] - cis @ cis[index]) / denominator if cis.shape[1] else kernel[:, index] / denominator
        cis = np.column_stack([cis, update])
        gains -= update ** 2
        gains[chosen] = -np.inf
    selected = [candidates[index] for index in chosen]
    selected_set = set(selected)
    return (selected + [mid for mid in ranked_mids if mid not in selected_set])[:k]


def tune_fusion_alpha(validation_context, semantic_cache, cf_score_cache):
    records = []
    for alpha in np.linspace(0.0, 1.0, 11):
        values = []
        for uid, truth in validation_context['ground_truth'].items():
            scores = fused_scores(uid, validation_context, semantic_cache.get(uid, {}), cf_score_cache.get(uid, {}), float(alpha), 'FusionFrozen')
            ranked = sorted(scores, key=scores.get, reverse=True)[:OUTPUT_LIST_K]
            values.append(ndcg_at_k(ranked, truth, 10))
        records.append({'alpha_semantic': float(alpha), 'beta_cf': float(1 - alpha), 'NDCG@10': float(np.mean(values)), 'n_users': len(values)})
    frame = pd.DataFrame(records)
    best = frame.sort_values(['NDCG@10', 'alpha_semantic'], ascending=[False, False]).iloc[0]
    frozen = {'alpha_semantic': float(best.alpha_semantic), 'beta_cf': float(best.beta_cf),
              'selection_metric': 'validation NDCG@10', 'validation_users': int(best.n_users)}
    frame.to_csv(EVAL_OUT / 'validation_fusion_grid.csv', index=False)
    (EVAL_OUT / 'frozen_fusion_weights.json').write_text(json.dumps(frozen, indent=2))
    return frozen


def generate_method_recommendations(context, stage, semantic_cache, rocchio_cache, include_itemknn=False):
    bundles = {'XSimGCL': load_embedding_bundle(stage, 'XSimGCL')}
    if stage == 'final':
        bundles.update({'BPR': load_embedding_bundle(stage, 'BPR'), 'LightGCN': load_embedding_bundle(stage, 'LightGCN')})
    cf_scores_by_model, cf_recs_by_model = {}, {}
    for name, bundle in bundles.items():
        cf_scores_by_model[name], cf_recs_by_model[name] = collaborative_outputs(bundle, context)
    methods = {name: recs for name, recs in cf_recs_by_model.items()}
    methods['MostPopular'] = {uid: [mid for mid in context['popularity_order'] if mid not in context['histories'][uid]][:OUTPUT_LIST_K] for uid in context['users']}
    methods['Random'] = {}
    candidate_array = np.array(context['candidate_ids'], dtype=np.int64)
    for uid in context['users']:
        available = candidate_array[~np.isin(candidate_array, np.fromiter(context['histories'][uid], dtype=np.int64))]
        rng = np.random.default_rng(SEED + uid)
        take = min(OUTPUT_LIST_K, len(available))
        methods['Random'][uid] = rng.choice(available, size=take, replace=False).astype(int).tolist() if take else []
    methods['SEM-RRF'] = {uid: sorted(semantic_cache.get(uid, {}), key=semantic_cache.get(uid, {}).get, reverse=True)[:OUTPUT_LIST_K] for uid in context['users']}
    methods['SEM-RRF+Rocchio'] = {uid: sorted(rocchio_cache.get(uid, {}), key=rocchio_cache.get(uid, {}).get, reverse=True)[:OUTPUT_LIST_K] for uid in context['users']}
    if include_itemknn and RUN_ITEMKNN:
        methods['ItemKNN'] = build_itemknn_recommendations(context)
    return methods, cf_scores_by_model['XSimGCL']


def append_fusion_methods(methods, context, semantic_cache, cf_scores, frozen):
    alpha = frozen['alpha_semantic']
    variants = ['FusionFrozen', 'FusionLanguageBridge', 'FullCurrentMetadata']
    score_maps = {variant: {} for variant in variants}
    for uid in context['users']:
        for variant in variants:
            score_maps[variant][uid] = fused_scores(uid, context, semantic_cache.get(uid, {}), cf_scores.get(uid, {}), alpha, variant)
            methods.setdefault(variant, {})[uid] = sorted(score_maps[variant][uid], key=score_maps[variant][uid].get, reverse=True)[:OUTPUT_LIST_K]
        bridge_ranked = sorted(score_maps['FusionLanguageBridge'][uid], key=score_maps['FusionLanguageBridge'][uid].get, reverse=True)[:200]
        methods.setdefault('FusionLanguageBridge+DPP', {})[uid] = greedy_dpp(bridge_ranked, score_maps['FusionLanguageBridge'][uid])
        full_ranked = sorted(score_maps['FullCurrentMetadata'][uid], key=score_maps['FullCurrentMetadata'][uid].get, reverse=True)[:200]
        gated = [mid for mid in full_ranked if quality_gate.get(mid, True)]
        gated_set = set(gated)
        gated += [mid for mid in full_ranked if mid not in gated_set]
        methods.setdefault('FullCurrentMetadata+Gate', {})[uid] = gated[:OUTPUT_LIST_K]
        methods.setdefault('FullCurrentMetadata+DPP', {})[uid] = greedy_dpp(full_ranked, score_maps['FullCurrentMetadata'][uid])
    return methods


In [11]:
# ── Development-only tuning, untouched final tests, and prespecified cohorts ──
validation_context = build_context('development', validation_users, validation_targets, development_candidates)
validation_semantic = build_semantic_cache(validation_context, 'validation_semantic_rrf', rocchio=False)
validation_rocchio = build_semantic_cache(validation_context, 'validation_semantic_rocchio', rocchio=True)
validation_methods, validation_xsim_scores = generate_method_recommendations(
    validation_context, 'development', validation_semantic, validation_rocchio, include_itemknn=False
)
frozen_weights = tune_fusion_alpha(validation_context, validation_semantic, validation_xsim_scores)
print('Frozen before final test:', frozen_weights)

# Primary known-item test.
test_context = build_context('final', test_users, test_targets, final_candidates)
test_semantic = build_semantic_cache(test_context, 'test_semantic_rrf', rocchio=False)
test_rocchio = build_semantic_cache(test_context, 'test_semantic_rocchio', rocchio=True)
test_methods, test_xsim_scores = generate_method_recommendations(
    test_context, 'final', test_semantic, test_rocchio, include_itemknn=True
)
test_methods = append_fusion_methods(test_methods, test_context, test_semantic, test_xsim_scores, frozen_weights)
primary_aggregate, primary_per_user = evaluate_recommendations(test_methods, test_context, 'primary_global_time_known_item')

# Balanced activity analysis is a subset of the untouched primary recommendations.
balanced_targets = test_targets.loc[test_targets.userId.isin(test_balanced_users)].copy()
balanced_context = build_context('final', test_balanced_users, balanced_targets, final_candidates)
balanced_methods = {method: {uid: recs.get(uid, []) for uid in test_balanced_users} for method, recs in test_methods.items()}
balanced_aggregate, balanced_per_user = evaluate_recommendations(balanced_methods, balanced_context, 'secondary_activity_balanced')

# Prespecified cross-language targets use the same lists as the primary test—no post-hoc reranking.
cross_users = sorted(map(int, cross_targets.userId.unique()))
cross_context = build_context('final', cross_users, cross_targets, final_candidates)
cross_methods = {method: {uid: recs.get(uid, []) for uid in cross_users} for method, recs in test_methods.items()}
cross_aggregate, cross_per_user = evaluate_recommendations(cross_methods, cross_context, 'secondary_cross_language')

# Candidate-cold test: candidates include pre-cutoff items plus the prespecified unseen positive targets.
cold_mapped_targets = test_cold_targets.loc[test_cold_targets.movieId.map(lambda mid: int(mid) in movie_to_semantic_row)].copy()
cold_users_mapped = sorted(map(int, cold_mapped_targets.userId.unique()))
cold_candidates = set(final_candidates) | set(map(int, cold_mapped_targets.movieId.unique()))
cold_context = build_context('final', cold_users_mapped, cold_mapped_targets, cold_candidates)
cold_semantic = build_semantic_cache(cold_context, 'cold_semantic_rrf', rocchio=False)
cold_rocchio = build_semantic_cache(cold_context, 'cold_semantic_rocchio', rocchio=True)
cold_methods, cold_xsim_scores = generate_method_recommendations(
    cold_context, 'final', cold_semantic, cold_rocchio, include_itemknn=False
)
cold_methods = append_fusion_methods(cold_methods, cold_context, cold_semantic, cold_xsim_scores, frozen_weights)
cold_aggregate, cold_per_user = evaluate_recommendations(cold_methods, cold_context, 'secondary_candidate_cold_start')

all_aggregate = pd.concat([primary_aggregate, balanced_aggregate, cross_aggregate, cold_aggregate], ignore_index=True)
all_per_user = pd.concat([primary_per_user, balanced_per_user, cross_per_user, cold_per_user], ignore_index=True)
all_aggregate.to_csv(EVAL_OUT / 'all_cohort_metrics.csv', index=False)
all_per_user.to_csv(EVAL_OUT / 'all_per_user_metrics.csv', index=False)
display(all_aggregate.sort_values(['cohort', 'NDCG@10'], ascending=[True, False]))


Frozen before final test: {'alpha_semantic': 0.2, 'beta_cf': 0.8, 'selection_metric': 'validation NDCG@10', 'validation_users': 2500}
Loaded completed ItemKNN cache: 2500 users
cold_semantic_rrf: 0/2389
cold_semantic_rrf: 100/2389
cold_semantic_rrf: 200/2389
cold_semantic_rrf: 300/2389
cold_semantic_rrf: 400/2389
cold_semantic_rrf: 500/2389
cold_semantic_rrf: 600/2389
cold_semantic_rrf: 700/2389
cold_semantic_rrf: 800/2389
cold_semantic_rrf: 900/2389
cold_semantic_rrf: 1000/2389
cold_semantic_rrf: 1100/2389
cold_semantic_rrf: 1200/2389
cold_semantic_rrf: 1300/2389
cold_semantic_rrf: 1400/2389
cold_semantic_rrf: 1500/2389
cold_semantic_rrf: 1600/2389
cold_semantic_rrf: 1700/2389
cold_semantic_rrf: 1800/2389
cold_semantic_rrf: 1900/2389
cold_semantic_rrf: 2000/2389
cold_semantic_rrf: 2100/2389
cold_semantic_rrf: 2200/2389
cold_semantic_rrf: 2300/2389
cold_semantic_rocchio: 0/2389
cold_semantic_rocchio: 100/2389
cold_semantic_rocchio: 200/2389
cold_semantic_rocchio: 300/2389
cold_semantic

,cohort,method,n_users,CatalogCoverage,targets,ILD@10,Novelty@10,LongTail@10,LanguageMapCoverage@10,NonDominantLanguageExposure@10,...,Precision@20,Hit@20,MRR@20,MAP@20,NDCG@50,Recall@50,Precision@50,Hit@50,MRR@50,MAP@50
8,primary_global_time_known_item,FusionFrozen,2500,0.087351,7.767600,0.729846,10.991142,0.000120,0.913480,0.093543,...,0.041400,0.470800,0.158816,0.035555,0.119221,0.194640,0.030072,0.651200,0.164570,0.041985
0,primary_global_time_known_item,XSimGCL,2500,0.085373,7.767600,0.732117,10.984561,0.000120,0.905040,0.094165,...,0.041460,0.473600,0.159129,0.035248,0.119060,0.195085,0.030008,0.643600,0.164615,0.041779
9,primary_global_time_known_item,FusionLanguageBridge,2500,0.131217,7.767600,0.728981,11.226082,0.018040,0.962880,0.106256,...,0.039460,0.457600,0.154045,0.034458,0.113604,0.183763,0.028312,0.628000,0.159523,0.040268
1,primary_global_time_known_item,BPR,2500,0.087032,7.767600,0.752662,10.119664,0.000080,0.923840,0.067158,...,0.038600,0.449200,0.152395,0.032734,0.111594,0.182612,0.028008,0.624000,0.158012,0.038610
7,primary_global_time_known_item,ItemKNN,2500,0.071162,7.767600,0.760529,10.125199,0.012240,0.930320,0.014669,...,0.039440,0.448000,0.145148,0.033520,0.109224,0.177204,0.027304,0.609200,0.150407,0.038984
12,primary_global_time_known_item,FullCurrentMetadata+Gate,2500,0.098291,7.767600,0.705615,11.586510,0.039600,0.992360,0.165785,...,0.038880,0.457200,0.142273,0.031281,0.110482,0.184890,0.028488,0.634800,0.148116,0.037195
11,primary_global_time_known_item,FusionLanguageBridge+DPP,2500,0.139905,7.767600,0.910599,12.333405,0.113440,0.941400,0.239513,...,0.030100,0.405200,0.147685,0.028501,0.092146,0.141705,0.021312,0.571600,0.152893,0.032087
10,primary_global_time_known_item,FullCurrentMetadata,2500,0.139525,7.767600,0.706988,13.635616,0.217960,0.995400,0.336899,...,0.033660,0.410400,0.131396,0.027958,0.102431,0.174037,0.026832,0.616800,0.138088,0.033759
13,primary_global_time_known_item,FullCurrentMetadata+DPP,2500,0.147726,7.767600,0.905580,14.274283,0.275120,0.991440,0.407979,...,0.026260,0.372000,0.123042,0.022152,0.080436,0.132138,0.019656,0.558000,0.129044,0.025554
3,primary_global_time_known_item,MostPopular,2500,0.013085,7.767600,0.798116,8.514434,0.000000,0.975480,0.011421,...,0.022560,0.288800,0.088958,0.018120,0.067217,0.114206,0.017568,0.439600,0.093741,0.021735


In [12]:
# ── Paired uncertainty, multiplicity correction, plots, and final manifest ─────
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt

PRESPECIFIED_COMPARISONS = {
    'primary_global_time_known_item': ('FusionFrozen', ['MostPopular', 'ItemKNN', 'BPR', 'LightGCN', 'XSimGCL', 'SEM-RRF']),
    'secondary_cross_language': ('FusionLanguageBridge', ['MostPopular', 'ItemKNN', 'BPR', 'XSimGCL', 'SEM-RRF']),
    'secondary_candidate_cold_start': ('SEM-RRF', ['MostPopular', 'BPR', 'LightGCN', 'XSimGCL']),
}

def paired_inference(frame, cohort, proposed, baselines, metric='NDCG@10'):
    subset = frame.loc[frame.cohort == cohort, ['userId', 'method', metric]]
    wide = subset.pivot_table(index='userId', columns='method', values=metric, aggfunc='first')
    rows = []
    rng = np.random.default_rng(SEED)
    for baseline in baselines:
        if proposed not in wide or baseline not in wide: continue
        paired = wide[[proposed, baseline]].dropna()
        differences = (paired[proposed] - paired[baseline]).to_numpy()
        if not len(differences): continue
        boot = np.empty(BOOTSTRAP_REPLICATES, dtype=float)
        for index in range(BOOTSTRAP_REPLICATES):
            boot[index] = differences[rng.integers(0, len(differences), len(differences))].mean()
        try:
            p_value = float(wilcoxon(differences, zero_method='zsplit', alternative='two-sided').pvalue)
        except ValueError:
            p_value = 1.0
        rows.append({'cohort': cohort, 'proposed': proposed, 'baseline': baseline, 'metric': metric,
                     'n_common_users': int(len(differences)), 'mean_paired_difference': float(differences.mean()),
                     'bootstrap_ci_low': float(np.quantile(boot, 0.025)),
                     'bootstrap_ci_high': float(np.quantile(boot, 0.975)), 'wilcoxon_p_raw': p_value})
    result = pd.DataFrame(rows)
    return result

inference_frames = []
for cohort, (proposed, baselines) in PRESPECIFIED_COMPARISONS.items():
    for metric in ['NDCG@10', 'Recall@10']:
        inference_frames.append(paired_inference(all_per_user, cohort, proposed, baselines, metric))
# Quantify the relevance/diversity change caused by DPP using the same user-level pairs.
for metric in ['NDCG@10', 'ILD@10']:
    inference_frames.append(paired_inference(
        all_per_user, 'primary_global_time_known_item',
        'FusionLanguageBridge+DPP', ['FusionLanguageBridge'], metric,
    ))
paired_results = pd.concat(inference_frames, ignore_index=True)
if not paired_results.empty:
    # Holm correction over the full prespecified family, not separately per convenient subset.
    order = np.argsort(paired_results.wilcoxon_p_raw.to_numpy())
    adjusted = np.empty(len(paired_results)); running = 0.0
    for rank, position in enumerate(order):
        candidate = min(1.0, paired_results.wilcoxon_p_raw.iloc[position] * (len(paired_results) - rank))
        running = max(running, candidate); adjusted[position] = running
    paired_results['wilcoxon_p_holm'] = adjusted
paired_results.to_csv(EVAL_OUT / 'prespecified_paired_inference.csv', index=False)
display(paired_results)

# Relevance–diversity Pareto plot. Current-metadata methods are visibly labeled as snapshot analyses.
primary_plot = all_aggregate.loc[all_aggregate.cohort == 'primary_global_time_known_item'].copy()
fig, ax = plt.subplots(figsize=(8, 5))
for _, row in primary_plot.iterrows():
    ax.scatter(row['NDCG@10'], row['ILD@10'], s=55)
    ax.annotate(row['method'], (row['NDCG@10'], row['ILD@10']), fontsize=7)
ax.set_xlabel('NDCG@10'); ax.set_ylabel('Intra-list genre diversity@10')
ax.set_title('Primary relevance–diversity trade-off')
ax.grid(alpha=0.25); fig.tight_layout()
fig.savefig(EVAL_OUT / 'relevance_diversity_pareto.png', dpi=220); plt.close(fig)

versions = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'torch': torch.__version__}
try:
    import recbole; versions['recbole'] = recbole.__version__
except Exception: pass
try:
    import faiss; versions['faiss'] = getattr(faiss, '__version__', 'unknown')
except Exception: pass

final_manifest = {
    'completed_utc': pd.Timestamp.utcnow().isoformat(),
    'protocol_manifest': manifest,
    'frozen_fusion_weights': frozen_weights,
    'prespecified_headline_methods': {
        'primary': 'FusionFrozen', 'cross_language_secondary': 'FusionLanguageBridge',
        'candidate_cold_secondary': 'SEM-RRF'},
    'metadata_warning': 'IMDb ratings/votes are a current snapshot and are not used for the primary headline temporal comparison.',
    'timestamp_warning': 'MovieLens timestamps are rating-event timestamps, not verified watch timestamps.',
    'versions': versions,
    'outputs': {path.name: sha256_file(path) for path in [
        EVAL_OUT / 'all_cohort_metrics.csv', EVAL_OUT / 'all_per_user_metrics.csv',
        EVAL_OUT / 'prespecified_paired_inference.csv', EVAL_OUT / 'frozen_fusion_weights.json']},
}
(EVAL_OUT / 'publication_run_manifest.json').write_text(json.dumps(final_manifest, indent=2))
print('Publication evaluation complete:', EVAL_OUT)
print('Return all_cohort_metrics.csv, prespecified_paired_inference.csv, frozen_fusion_weights.json, and publication_run_manifest.json for manuscript revision.')


,cohort,proposed,baseline,metric,n_common_users,mean_paired_difference,bootstrap_ci_low,bootstrap_ci_high,wilcoxon_p_raw,wilcoxon_p_holm
0,primary_global_time_known_item,FusionFrozen,MostPopular,NDCG@10,2500,0.031711,0.026347,0.037091,8.205240e-32,2.133362e-30
1,primary_global_time_known_item,FusionFrozen,ItemKNN,NDCG@10,2500,0.005457,0.000031,0.011025,6.562411e-02,5.972679e-01
2,primary_global_time_known_item,FusionFrozen,BPR,NDCG@10,2500,0.003741,-0.000771,0.008277,1.467900e-02,2.495431e-01
3,primary_global_time_known_item,FusionFrozen,LightGCN,NDCG@10,2500,0.033191,0.027913,0.038601,3.545793e-35,9.928219e-34
4,primary_global_time_known_item,FusionFrozen,XSimGCL,NDCG@10,2500,0.000339,-0.000707,0.001382,5.045182e-03,9.081328e-02
5,primary_global_time_known_item,FusionFrozen,SEM-RRF,NDCG@10,2500,0.061606,0.056715,0.066723,7.170095e-119,2.151028e-117
6,primary_global_time_known_item,FusionFrozen,MostPopular,Recall@10,2500,0.033797,0.027891,0.039728,4.807278e-34,1.297965e-32
7,primary_global_time_known_item,FusionFrozen,ItemKNN,Recall@10,2500,0.007297,0.001176,0.013432,5.027773e-02,5.972679e-01
8,primary_global_time_known_item,FusionFrozen,BPR,Recall@10,2500,0.003615,-0.001947,0.009185,6.156079e-02,5.972679e-01
9,primary_global_time_known_item,FusionFrozen,LightGCN,Recall@10,2500,0.035139,0.029308,0.041231,1.352445e-36,3.922090e-35


Publication evaluation complete: /content/drive/MyDrive/cinematch/outputs/publication_audit/publication_eval
Return all_cohort_metrics.csv, prespecified_paired_inference.csv, frozen_fusion_weights.json, and publication_run_manifest.json for manuscript revision.


## What to return after the run

Upload or attach these four files from `outputs/publication_audit/publication_eval/`:

1. `all_cohort_metrics.csv`
2. `prespecified_paired_inference.csv`
3. `frozen_fusion_weights.json`
4. `publication_run_manifest.json`

Also include the Colab error traceback if any cell fails. Do not change the cohort, targets, cutoffs, fusion grid, or headline methods after seeing test results. A result is publishable because it is reproducible and falsifiable—not because every metric favors CineMatch.

Interpretation rules:

- Primary claim: global-time known-item performance of `FusionFrozen` against the declared baselines.
- Cross-language claim: secondary performance of `FusionLanguageBridge`, with eligible user count, target count, dominance threshold, and language-mapping coverage.
- Item cold-start claim: secondary zero-shot performance of `SEM-RRF`; report how many cold targets were mappable to the semantic index.
- Relevance/diversity claim: paired NDCG/Recall together with ILD, catalog coverage, novelty, and long-tail exposure.
- Current IMDb quality/gate rows are descriptive snapshot ablations, not leak-free historical evidence.
- MovieLens provides no culture, nationality, preferred-language, or satisfaction label. Use “cross-language title retrieval,” never “proved cross-cultural satisfaction.”
